# 01. Multi-horizon TP probability 전처리

완료된 `t`봉까지의 30개 1분봉으로, `first_ask[t+1]` 진입 후 미래 1~5분의
보수적 net +3% 도달 시점을 만든다.

- 각 판단 시점은 자신의 진입 ask와 미래 quote로 독립 재계산한다.
- 유효한 인접 positive를 모두 유지하며 중복 제거·episode 역가중을 하지 않는다.
- `target_net3_by_1m`부터 `target_net3_by_5m`까지 누적 target을 저장한다.
- discrete-time hazard 학습용 최초 도달 event와 at-risk mask를 함께 저장한다.
- `t+1` 이후의 OHLCV/quote는 feature가 아니라 label과 실행 수익 경로에만 사용한다.
- 최종 확률 모델은 class-weighted sigmoid가 아니라 자연 발생 비율의 proper loss와
  날짜 OOF calibration을 사용한다.

In [1]:
from pathlib import Path
import json
import math
import time

import numpy as np
import pandas as pd


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
DATA_ROOT = (PROJECT_ROOT / '../../data/stock_data/raw').resolve()
RESULT_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

VERSION = 'scalp_30m_ohlcv_net3_multihorizon_5m_v2'
SEQUENCE_LENGTH = 30
HORIZON_MINUTES = 5
NET_TARGET_RETURN = 0.03
STAKE_USD = 100.0
COMMISSION_USD_PER_SIDE = 0.0
SEC_SECTION31_RATE = 20.60 / 1_000_000
FINRA_TAF_PER_SHARE = 0.000166
FINRA_TAF_CAP_USD = 8.30

SOURCE_FILES = sorted(DATA_ROOT.rglob('*_enriched.csv'))
assert SOURCE_FILES, DATA_ROOT
print('project:', PROJECT_ROOT)
print('source files:', len(SOURCE_FILES))
print('result root:', RESULT_ROOT)

project: /home/user/urbandatalab/YSLee/code/Detecting-entry-signals-for-short-term-trades-right-before-a-rapid-price-surge
source files: 208
result root: /home/user/urbandatalab/YSLee/results/preprocessing


## Feature 정의

현재 봉의 거래량 급증은 현재 봉을 제외한 이전 5/10/20봉 median과 비교한다. 공격적 매수·매도 거래량은 총 거래량에 대한 비율로 바꿔 종목 크기의 영향을 줄인다.

In [2]:
REQUIRED_COLUMNS = [
    'symbol', 'timestamp_utc', 'open', 'high', 'low', 'close',
    'volume', 'trade_count', 'alpaca_vwap',
    'first_bid', 'first_ask', 'last_bid', 'min_bid',
    'aggressive_buy_volume', 'aggressive_sell_volume', 'unknown_trade_volume',
]

PRICE_FEATURES = [
    'log_open_to_previous_close', 'log_high_to_open', 'log_low_to_open',
    'log_close_to_open', 'log_return_1', 'log_range',
    'upper_wick_ratio', 'lower_wick_ratio', 'close_location',
]
ABSOLUTE_VOLUME_FEATURES = ['log_volume']
RELATIVE_FLOW_FEATURES = [
    'log_relative_volume_5', 'log_relative_volume_10', 'log_relative_volume_20',
    'robust_log_volume_z20', 'volume_ema3_to_ema10',
    'log_relative_notional_20', 'log_relative_trade_count_20',
    'log_relative_average_trade_size_20',
    'aggressive_net_flow', 'aggressive_participation', 'aggressive_net_flow_3',
    'signed_log_relative_aggressive_net_notional_20',
    'log_relative_aggressive_gross_notional_20',
    'return_x_log_relative_volume_20',
]
LOCATION_FEATURES = [
    'close_to_bar_vwap', 'close_to_session_vwap',
    'close_to_ema5', 'close_to_ema9', 'close_to_ema20',
    'close_to_session_high', 'close_to_session_low',
    'close_to_prior_session_high', 'close_to_prior_session_low',
    'close_to_high_5', 'close_to_low_5',
    'close_to_high_10', 'close_to_low_10',
    'close_to_high_20', 'close_to_low_20',
]
FEATURE_NAMES = PRICE_FEATURES + ABSOLUTE_VOLUME_FEATURES + RELATIVE_FLOW_FEATURES + LOCATION_FEATURES
DEFAULT_FEATURE_NAMES = [name for name in FEATURE_NAMES if name not in ABSOLUTE_VOLUME_FEATURES]


def safe_log_ratio(numerator, denominator, epsilon=1e-8):
    return np.log(np.clip(numerator, epsilon, None) / np.clip(denominator, epsilon, None))


def prior_median(values, window):
    return values.shift(1).rolling(window, min_periods=window).median()


def relative_log(values, window):
    baseline = prior_median(values, window)
    return np.log((values.clip(lower=0) + 1.0) / (baseline.clip(lower=0) + 1.0))


def build_features(frame):
    frame = frame.copy()
    open_price = frame['open'].astype(float)
    high = frame['high'].astype(float)
    low = frame['low'].astype(float)
    close = frame['close'].astype(float)
    volume = frame['volume'].astype(float).clip(lower=0)
    trade_count = frame['trade_count'].astype(float).clip(lower=0)
    bar_vwap = frame['alpaca_vwap'].astype(float)
    previous_close = close.shift(1)

    features = pd.DataFrame(index=frame.index)
    features['log_open_to_previous_close'] = safe_log_ratio(open_price, previous_close)
    features['log_high_to_open'] = safe_log_ratio(high, open_price)
    features['log_low_to_open'] = safe_log_ratio(low, open_price)
    features['log_close_to_open'] = safe_log_ratio(close, open_price)
    features['log_return_1'] = safe_log_ratio(close, previous_close)
    features['log_range'] = safe_log_ratio(high, low)
    features['upper_wick_ratio'] = (high - np.maximum(open_price, close)) / open_price.clip(lower=1e-8)
    features['lower_wick_ratio'] = (np.minimum(open_price, close) - low) / open_price.clip(lower=1e-8)
    features['close_location'] = (close - low) / (high - low).replace(0, np.nan)
    features['close_location'] = features['close_location'].fillna(0.5)

    log_volume = np.log1p(volume)
    features['log_volume'] = log_volume
    for window in [5, 10, 20]:
        features[f'log_relative_volume_{window}'] = relative_log(volume, window)
    prior_log_median = log_volume.shift(1).rolling(20, min_periods=20).median()
    prior_log_q25 = log_volume.shift(1).rolling(20, min_periods=20).quantile(0.25)
    prior_log_q75 = log_volume.shift(1).rolling(20, min_periods=20).quantile(0.75)
    robust_scale = (prior_log_q75 - prior_log_q25).clip(lower=0.05)
    features['robust_log_volume_z20'] = (log_volume - prior_log_median) / robust_scale
    volume_ema3 = volume.ewm(span=3, adjust=False).mean()
    volume_ema10 = volume.ewm(span=10, adjust=False).mean()
    features['volume_ema3_to_ema10'] = safe_log_ratio(volume_ema3 + 1, volume_ema10 + 1)

    notional = bar_vwap * volume
    average_trade_size = volume / trade_count.clip(lower=1)
    features['log_relative_notional_20'] = relative_log(notional, 20)
    features['log_relative_trade_count_20'] = relative_log(trade_count, 20)
    features['log_relative_average_trade_size_20'] = relative_log(average_trade_size, 20)

    aggressive_buy = frame['aggressive_buy_volume'].astype(float).clip(lower=0)
    aggressive_sell = frame['aggressive_sell_volume'].astype(float).clip(lower=0)
    classified_volume = aggressive_buy + aggressive_sell
    features['aggressive_net_flow'] = ((aggressive_buy - aggressive_sell) / classified_volume.replace(0, np.nan)).fillna(0)
    features['aggressive_participation'] = (classified_volume / volume.replace(0, np.nan)).fillna(0).clip(0, 1)
    flow3 = (aggressive_buy - aggressive_sell).rolling(3, min_periods=1).sum()
    classified3 = classified_volume.rolling(3, min_periods=1).sum()
    features['aggressive_net_flow_3'] = (flow3 / classified3.replace(0, np.nan)).fillna(0)
    prior_notional_median20 = prior_median(notional, 20).clip(lower=1e-8)
    aggressive_net_notional = bar_vwap * (aggressive_buy - aggressive_sell)
    aggressive_gross_notional = bar_vwap * classified_volume
    relative_net_notional = aggressive_net_notional / prior_notional_median20
    features['signed_log_relative_aggressive_net_notional_20'] = (
        np.sign(relative_net_notional) * np.log1p(relative_net_notional.abs())
    )
    features['log_relative_aggressive_gross_notional_20'] = np.log1p(
        aggressive_gross_notional / prior_notional_median20
    )
    features['return_x_log_relative_volume_20'] = (
        features['log_return_1'] * features['log_relative_volume_20']
    )

    cumulative_volume = volume.cumsum()
    session_vwap = (bar_vwap * volume).cumsum() / cumulative_volume.replace(0, np.nan)
    features['close_to_bar_vwap'] = close / bar_vwap.clip(lower=1e-8) - 1
    features['close_to_session_vwap'] = close / session_vwap.clip(lower=1e-8) - 1
    for span in [5, 9, 20]:
        ema = close.ewm(span=span, adjust=False).mean()
        features[f'close_to_ema{span}'] = close / ema.clip(lower=1e-8) - 1

    session_high = high.cummax()
    session_low = low.cummin()
    prior_session_high = high.shift(1).cummax()
    prior_session_low = low.shift(1).cummin()
    features['close_to_session_high'] = close / session_high.clip(lower=1e-8) - 1
    features['close_to_session_low'] = close / session_low.clip(lower=1e-8) - 1
    features['close_to_prior_session_high'] = close / prior_session_high.clip(lower=1e-8) - 1
    features['close_to_prior_session_low'] = close / prior_session_low.clip(lower=1e-8) - 1
    for window in [5, 10, 20]:
        rolling_high = high.rolling(window, min_periods=window).max()
        rolling_low = low.rolling(window, min_periods=window).min()
        features[f'close_to_high_{window}'] = close / rolling_high.clip(lower=1e-8) - 1
        features[f'close_to_low_{window}'] = close / rolling_low.clip(lower=1e-8) - 1

    return features[FEATURE_NAMES].astype(np.float32)


print('feature count:', len(FEATURE_NAMES))
print('default feature count:', len(DEFAULT_FEATURE_NAMES))

feature count: 39
default feature count: 38


## 실행 가능 multi-horizon target

진입은 `first_ask[t+1]`, TP 판정은 각 미래 완성봉의 가장 불리한 `min_bid`를 사용한다.
예를 들어 3분에 최초 TP라면 누적 target은 `[0, 0, 1, 1, 1]`이다.

hazard target은 최초 도달 분에만 event=1이고 그 시점까지 at-risk=1이다. 미도달 표본은
5개 horizon 모두 event=0, at-risk=1이다. 각 분의 `first_bid`, `min_bid`, `last_bid`
실행 수익률도 저장하되 미래 값은 절대 feature로 사용하지 않는다.

In [3]:
def executable_net_return(entry_ask, exit_bid):
    shares = STAKE_USD / entry_ask
    exit_notional = shares * exit_bid
    section31 = exit_notional * SEC_SECTION31_RATE
    taf = 0.0 if exit_bid < FINRA_TAF_PER_SHARE else min(shares * FINRA_TAF_PER_SHARE, FINRA_TAF_CAP_USD)
    total_cost = 2 * COMMISSION_USD_PER_SIDE + section31 + taf
    return (exit_notional - STAKE_USD - total_cost) / STAKE_USD


def load_source(path):
    frame = pd.read_csv(path, usecols=REQUIRED_COLUMNS)
    frame['timestamp_utc'] = pd.to_datetime(frame['timestamp_utc'], utc=True, errors='coerce')
    numeric_columns = [column for column in REQUIRED_COLUMNS if column not in {'symbol', 'timestamp_utc'}]
    frame[numeric_columns] = frame[numeric_columns].apply(pd.to_numeric, errors='coerce')
    return frame.sort_values('timestamp_utc').reset_index(drop=True)


HORIZON_TARGET_COLUMNS = [f'target_net3_by_{horizon}m' for horizon in range(1, HORIZON_MINUTES + 1)]
AT_HORIZON_TARGET_COLUMNS = [f'target_net3_at_{horizon}m' for horizon in range(1, HORIZON_MINUTES + 1)]
HAZARD_EVENT_COLUMNS = [f'hazard_event_{horizon}m' for horizon in range(1, HORIZON_MINUTES + 1)]
HAZARD_AT_RISK_COLUMNS = [f'hazard_at_risk_{horizon}m' for horizon in range(1, HORIZON_MINUTES + 1)]

sequences = []
metadata_records = []
quality_records = []
started = time.time()
sample_index = 0

for file_number, path in enumerate(SOURCE_FILES, start=1):
    frame = load_source(path)
    features = build_features(frame)
    timestamps = frame['timestamp_utc']
    minute_number = timestamps.astype('int64').to_numpy() // 60_000_000_000
    run_number = np.r_[0, np.cumsum(np.diff(minute_number) != 1)]
    feature_matrix = features.to_numpy(np.float32)
    total_candidates = valid_candidates = 0
    rejected_gap = rejected_feature = rejected_quote = 0

    for decision_index in range(SEQUENCE_LENGTH - 1, len(frame) - HORIZON_MINUTES):
        total_candidates += 1
        full_start = decision_index - SEQUENCE_LENGTH + 1
        full_end = decision_index + HORIZON_MINUTES + 1
        if not np.all(np.diff(minute_number[full_start:full_end]) == 1):
            rejected_gap += 1
            continue
        sequence = feature_matrix[full_start:decision_index + 1]
        if sequence.shape != (SEQUENCE_LENGTH, len(FEATURE_NAMES)) or not np.isfinite(sequence).all():
            rejected_feature += 1
            continue

        entry_index = decision_index + 1
        future_indices = np.arange(entry_index, entry_index + HORIZON_MINUTES)
        entry_ask = float(frame.at[entry_index, 'first_ask'])
        future_first_bid = frame.loc[future_indices, 'first_bid'].to_numpy(float)
        future_min_bid = frame.loc[future_indices, 'min_bid'].to_numpy(float)
        future_last_bid = frame.loc[future_indices, 'last_bid'].to_numpy(float)
        all_quotes = np.r_[entry_ask, future_first_bid, future_min_bid, future_last_bid]
        if not np.isfinite(all_quotes).all() or np.any(all_quotes <= 0):
            rejected_quote += 1
            continue

        first_bid_returns = np.asarray([
            executable_net_return(entry_ask, exit_bid) for exit_bid in future_first_bid
        ], dtype=np.float32)
        min_bid_returns = np.asarray([
            executable_net_return(entry_ask, exit_bid) for exit_bid in future_min_bid
        ], dtype=np.float32)
        last_bid_returns = np.asarray([
            executable_net_return(entry_ask, exit_bid) for exit_bid in future_last_bid
        ], dtype=np.float32)
        reached_at = min_bid_returns >= NET_TARGET_RETURN
        reached_by = np.maximum.accumulate(reached_at).astype(np.int8)
        first_hit_minute = int(np.flatnonzero(reached_at)[0] + 1) if reached_at.any() else 0
        decision_bar_timestamp = timestamps.iloc[decision_index]
        decision_available_timestamp = decision_bar_timestamp + pd.Timedelta(minutes=1)

        sequences.append(sequence.copy())
        record = {
            'sample_id': f'{VERSION}_{sample_index:08d}',
            'feature_index': sample_index,
            'session': path.parent.name,
            'symbol': str(frame.at[decision_index, 'symbol']).upper(),
            'source_file': str(path.resolve()),
            'run_id': f'{path.stem}::{int(run_number[decision_index])}',
            'input_start_timestamp': timestamps.iloc[full_start],
            'decision_bar_timestamp': decision_bar_timestamp,
            'decision_available_timestamp': decision_available_timestamp,
            'entry_timestamp': timestamps.iloc[entry_index],
            'entry_ask': entry_ask,
            'target_net3_within_5m': int(reached_at.any()),
            'first_hit_minute': first_hit_minute,
            'first_hit_class': first_hit_minute,
            'best_min_bid_net_return_1_5m': float(min_bid_returns.max()),
            'worst_min_bid_net_return_1_5m': float(min_bid_returns.min()),
            'timeout_bid_5m': float(future_last_bid[-1]),
            'timeout_net_return_5m': float(last_bid_returns[-1]),
        }
        for position, horizon in enumerate(range(1, HORIZON_MINUTES + 1)):
            at_risk = int(first_hit_minute == 0 or horizon <= first_hit_minute)
            record[HORIZON_TARGET_COLUMNS[position]] = int(reached_by[position])
            record[AT_HORIZON_TARGET_COLUMNS[position]] = int(reached_at[position])
            record[HAZARD_EVENT_COLUMNS[position]] = int(first_hit_minute == horizon)
            record[HAZARD_AT_RISK_COLUMNS[position]] = at_risk
            record[f'future_timestamp_{horizon}'] = timestamps.iloc[future_indices[position]]
            record[f'first_bid_{horizon}'] = float(future_first_bid[position])
            record[f'min_bid_{horizon}'] = float(future_min_bid[position])
            record[f'last_bid_{horizon}'] = float(future_last_bid[position])
            record[f'net_return_first_bid_{horizon}'] = float(first_bid_returns[position])
            record[f'net_return_min_bid_{horizon}'] = float(min_bid_returns[position])
            record[f'net_return_last_bid_{horizon}'] = float(last_bid_returns[position])
        metadata_records.append(record)
        sample_index += 1
        valid_candidates += 1

    quality_records.append({
        'session': path.parent.name, 'source_file': str(path.resolve()),
        'symbol': str(frame['symbol'].iloc[0]).upper() if len(frame) else '',
        'rows': len(frame), 'total_candidates': total_candidates,
        'valid_candidates': valid_candidates, 'rejected_gap': rejected_gap,
        'rejected_feature': rejected_feature, 'rejected_quote': rejected_quote,
    })
    if file_number % 25 == 0 or file_number == len(SOURCE_FILES):
        print(f'{file_number:>3}/{len(SOURCE_FILES)} files | samples={sample_index:,}')

print(f'elapsed: {(time.time() - started) / 60:.2f} min')

 25/208 files | samples=9,116


 50/208 files | samples=17,281


 75/208 files | samples=26,050


100/208 files | samples=33,019


125/208 files | samples=42,705


150/208 files | samples=51,108


175/208 files | samples=61,498


200/208 files | samples=69,960


208/208 files | samples=72,181
elapsed: 0.87 min


## Positive episode 기록과 artifact 저장

인접 positive는 모두 유지한다. `positive_episode_id`는 한 급등이 만든 연속 positive의
길이와 평가 중복을 확인하기 위한 메타데이터일 뿐, 삭제나 loss 역가중에 사용하지 않는다.

In [4]:
sequence_array = np.stack(sequences).astype(np.float32)
metadata = pd.DataFrame(metadata_records).sort_values(
    ['session', 'symbol', 'decision_bar_timestamp']
).reset_index(drop=True)
quality = pd.DataFrame(quality_records)

# 정렬 뒤 feature row와 metadata row가 동일하도록 sequence도 다시 정렬한다.
feature_order = metadata['feature_index'].to_numpy(np.int64)
sequence_array = sequence_array[feature_order]
metadata['feature_index'] = np.arange(len(metadata), dtype=np.int64)

metadata['positive_episode_id'] = pd.Series(pd.NA, index=metadata.index, dtype='string')
metadata['positive_episode_position'] = np.zeros(len(metadata), dtype=np.int16)
metadata['positive_episode_length'] = np.zeros(len(metadata), dtype=np.int16)
episode_records = []
episode_number = 0
positive_mask = metadata['target_net3_within_5m'].eq(1)
for (session, symbol, run_id), group in metadata.loc[positive_mask].groupby(
    ['session', 'symbol', 'run_id'], sort=False
):
    group = group.sort_values('decision_bar_timestamp')
    minute_gap = group['decision_bar_timestamp'].diff().dt.total_seconds().div(60)
    local_episode = minute_gap.ne(1).cumsum()
    for _, episode in group.groupby(local_episode, sort=False):
        episode_id = f'{VERSION}_episode_{episode_number:06d}'
        positions = np.arange(1, len(episode) + 1, dtype=np.int16)
        metadata.loc[episode.index, 'positive_episode_id'] = episode_id
        metadata.loc[episode.index, 'positive_episode_position'] = positions
        metadata.loc[episode.index, 'positive_episode_length'] = len(episode)
        episode_records.append({
            'positive_episode_id': episode_id, 'session': session, 'symbol': symbol,
            'run_id': run_id, 'start_timestamp': episode['decision_bar_timestamp'].iloc[0],
            'end_timestamp': episode['decision_bar_timestamp'].iloc[-1],
            'positive_samples': len(episode),
            'earliest_first_hit_minute': int(episode['first_hit_minute'].min()),
        })
        episode_number += 1
episodes = pd.DataFrame(episode_records)

horizon_target_array = metadata[HORIZON_TARGET_COLUMNS].to_numpy(np.int8)
at_horizon_target_array = metadata[AT_HORIZON_TARGET_COLUMNS].to_numpy(np.int8)
hazard_event_array = metadata[HAZARD_EVENT_COLUMNS].to_numpy(np.int8)
hazard_at_risk_array = metadata[HAZARD_AT_RISK_COLUMNS].to_numpy(np.int8)

assert len(sequence_array) == len(metadata)
assert sequence_array.shape[1:] == (SEQUENCE_LENGTH, len(FEATURE_NAMES))
assert np.isfinite(sequence_array).all()
assert np.array_equal(metadata['feature_index'].to_numpy(), np.arange(len(metadata)))
assert (metadata['decision_available_timestamp'] == metadata['entry_timestamp']).all()
assert (np.diff(horizon_target_array, axis=1) >= 0).all()
assert np.array_equal(horizon_target_array, np.maximum.accumulate(at_horizon_target_array, axis=1))
assert (hazard_event_array.sum(axis=1) <= 1).all()
assert np.array_equal(hazard_event_array.sum(axis=1), metadata['target_net3_within_5m'].to_numpy())
assert int(positive_mask.sum()) == int(metadata['positive_episode_length'].gt(0).sum())

feature_path = RESULT_ROOT / f'{VERSION}_features.npz'
metadata_path = RESULT_ROOT / f'{VERSION}_metadata.parquet'
quality_path = RESULT_ROOT / f'{VERSION}_quality.parquet'
distribution_path = RESULT_ROOT / f'{VERSION}_label_distribution.parquet'
episode_path = RESULT_ROOT / f'{VERSION}_positive_episodes.parquet'
schema_path = RESULT_ROOT / f'{VERSION}_schema.json'

np.savez_compressed(
    feature_path, sequence=sequence_array, horizon_targets=horizon_target_array,
    at_horizon_targets=at_horizon_target_array,
    hazard_events=hazard_event_array, hazard_at_risk=hazard_at_risk_array,
)
metadata.to_parquet(metadata_path, index=False, compression='zstd')
quality.to_parquet(quality_path, index=False, compression='zstd')
episodes.to_parquet(episode_path, index=False, compression='zstd')
distribution = metadata.groupby('session').agg(
    samples=('sample_id', 'size'), symbols=('symbol', 'nunique'),
    positives_1m=('target_net3_by_1m', 'sum'), positive_rate_1m=('target_net3_by_1m', 'mean'),
    positives_2m=('target_net3_by_2m', 'sum'), positive_rate_2m=('target_net3_by_2m', 'mean'),
    positives_3m=('target_net3_by_3m', 'sum'), positive_rate_3m=('target_net3_by_3m', 'mean'),
    positives_4m=('target_net3_by_4m', 'sum'), positive_rate_4m=('target_net3_by_4m', 'mean'),
    positives_5m=('target_net3_by_5m', 'sum'), positive_rate_5m=('target_net3_by_5m', 'mean'),
    mean_best_min_bid_net_return=('best_min_bid_net_return_1_5m', 'mean'),
    mean_timeout_net_return=('timeout_net_return_5m', 'mean'),
).reset_index()
distribution.to_parquet(distribution_path, index=False, compression='zstd')

schema = {
    'version': VERSION, 'source_glob': str(DATA_ROOT / 'session_*/*_enriched.csv'),
    'sequence_length': SEQUENCE_LENGTH, 'horizon_minutes': HORIZON_MINUTES,
    'feature_names': FEATURE_NAMES, 'default_feature_names': DEFAULT_FEATURE_NAMES,
    'ablation_only_features': ABSOLUTE_VOLUME_FEATURES,
    'feature_groups': {
        'price': PRICE_FEATURES, 'absolute_volume': ABSOLUTE_VOLUME_FEATURES,
        'relative_flow': RELATIVE_FLOW_FEATURES, 'location': LOCATION_FEATURES,
    },
    'timing': {
        'decision': 'immediately after completed t bar', 'entry': 'first_ask[t+1]',
        'feature_cutoff': 'all sequence features end at completed t bar',
        'future_data_usage': 'future quotes are targets/diagnostics only and never model features',
    },
    'label': {
        'target_return_net': NET_TARGET_RETURN,
        'horizon_target_columns': HORIZON_TARGET_COLUMNS,
        'at_horizon_target_columns': AT_HORIZON_TARGET_COLUMNS,
        'probability_model_horizons': [2, 3, 4, 5],
        'structural_zero_horizons': [1],
        'hazard_event_columns': HAZARD_EVENT_COLUMNS,
        'hazard_at_risk_columns': HAZARD_AT_RISK_COLUMNS,
        'first_hit_class': '0=no hit, 1..5=first completed-minute min_bid net +3% hit',
        'positive_rule': 'cumulative any-hit by each horizon using min_bid and all execution costs',
        'adjacent_positive_policy': 'retain every independently valid decision timestamp without deduplication or inverse weighting',
    },
    'recommended_probability_training': {
        'output': 'four discrete-time hazard probabilities for minutes 2, 3, 4, 5; minute 1 is audit-only',
        'loss': 'unweighted masked binary cross-entropy on natural prevalence',
        'cumulative_probability': '1 - cumulative product of (1 - hazard) across minutes 2 through 5',
        'calibration': 'chronological OOF calibration only',
    },
    'execution_costs': {
        'stake_usd': STAKE_USD, 'commission_usd_per_side': COMMISSION_USD_PER_SIDE,
        'section31_rate_per_sold_dollar': SEC_SECTION31_RATE,
        'finra_taf_per_sold_share': FINRA_TAF_PER_SHARE,
        'finra_taf_cap_usd': FINRA_TAF_CAP_USD, 'spread': 'entry ask and exit bid',
    },
    'scaling': 'not applied; fit on each future Train fold only',
    'artifacts': {
        'features': str(feature_path), 'metadata': str(metadata_path),
        'quality': str(quality_path), 'label_distribution': str(distribution_path),
        'positive_episodes': str(episode_path),
    },
}
schema_path.write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')

print('sequence:', sequence_array.shape)
print('multi-horizon cumulative/at target:', horizon_target_array.shape, at_horizon_target_array.shape)
print('metadata:', metadata.shape)
print('positive samples retained:', int(metadata['target_net3_within_5m'].sum()))
print('positive episodes:', len(episodes))
display(distribution)
display(quality[['rows', 'total_candidates', 'valid_candidates', 'rejected_gap', 'rejected_feature', 'rejected_quote']].sum().to_frame('count'))

sequence: (72181, 30, 39)
multi-horizon cumulative/at target: (72181, 5) (72181, 5)
metadata: (72181, 76)
positive samples retained: 2847
positive episodes: 1128


,session,samples,symbols,positives_1m,positive_rate_1m,positives_2m,positive_rate_2m,positives_3m,positive_rate_3m,positives_4m,positive_rate_4m,positives_5m,positive_rate_5m,mean_best_min_bid_net_return,mean_timeout_net_return
0,session_2026-07-07,4321,13,0,0.0,12,0.002777,39,0.009026,79,0.018283,115,0.026614,-0.002089,-0.004694
1,session_2026-07-08,10343,25,0,0.0,38,0.003674,125,0.012085,247,0.023881,362,0.035000,-0.003138,-0.007469
2,session_2026-07-09,10208,27,0,0.0,64,0.006270,173,0.016947,275,0.026940,388,0.038009,-0.001989,-0.006434
3,session_2026-07-10,10230,29,0,0.0,90,0.008798,239,0.023363,384,0.037537,534,0.052199,-0.003133,-0.008216
4,session_2026-07-13,7953,18,0,0.0,69,0.008676,192,0.024142,304,0.038225,405,0.050924,-0.001929,-0.006242
5,session_2026-07-14,7043,20,0,0.0,46,0.006531,119,0.016896,199,0.028255,266,0.037768,-0.003032,-0.008038
6,session_2026-07-15,9556,21,0,0.0,57,0.005965,156,0.016325,287,0.030033,409,0.042800,-0.002456,-0.007069
7,session_2026-07-16,5918,16,0,0.0,27,0.004562,90,0.015208,161,0.027205,237,0.040047,-0.001598,-0.005094
8,session_2026-07-17,6609,20,0,0.0,10,0.001513,46,0.006960,87,0.013164,131,0.019821,-0.003383,-0.006528


,count
rows,127423
total_candidates,120351
valid_candidates,72181
rejected_gap,45420
rejected_feature,1276
rejected_quote,1474


In [5]:
horizon_summary = pd.DataFrame({
    'horizon_minute': np.arange(1, HORIZON_MINUTES + 1),
    'positives': horizon_target_array.sum(axis=0),
    'positive_rate': horizon_target_array.mean(axis=0),
    'first_hits': hazard_event_array.sum(axis=0),
    'at_risk': hazard_at_risk_array.sum(axis=0),
})
horizon_summary['empirical_hazard'] = horizon_summary['first_hits'] / horizon_summary['at_risk']

positive_count = int(metadata['target_net3_within_5m'].sum())
episode_count = len(episodes)
adjacent_positive_count = int((metadata['positive_episode_length'] > 1).sum())
label_summary = pd.Series({
    'samples': len(metadata), 'symbols': metadata['symbol'].nunique(),
    'sessions': metadata['session'].nunique(), 'positive_samples_5m': positive_count,
    'positive_rate_5m': positive_count / len(metadata), 'positive_episodes': episode_count,
    'mean_positive_samples_per_episode': positive_count / max(episode_count, 1),
    'positive_samples_in_multirow_episode': adjacent_positive_count,
    'share_positive_samples_in_multirow_episode': adjacent_positive_count / max(positive_count, 1),
    'max_positive_episode_length': int(episodes['positive_samples'].max()) if len(episodes) else 0,
    'median_first_hit_positive': metadata.loc[metadata['target_net3_within_5m'].eq(1), 'first_hit_minute'].median(),
})
display(label_summary.to_frame('value'))
display(horizon_summary)
display(episodes['positive_samples'].describe().to_frame('episode_length'))

,value
samples,72181.000000
symbols,111.000000
sessions,9.000000
positive_samples_5m,2847.000000
positive_rate_5m,0.039443
positive_episodes,1128.000000
mean_positive_samples_per_episode,2.523936
positive_samples_in_multirow_episode,2420.000000
share_positive_samples_in_multirow_episode,0.850018
max_positive_episode_length,11.000000


,horizon_minute,positives,positive_rate,first_hits,at_risk,empirical_hazard
0,1,0,0.000000,0,72181,0.000000
1,2,413,0.005722,413,72181,0.005722
2,3,1179,0.016334,766,71768,0.010673
3,4,2023,0.028027,844,71002,0.011887
4,5,2847,0.039443,824,70158,0.011745


,episode_length
count,1128.000000
mean,2.523936
std,1.637882
min,1.000000
25%,1.000000
50%,2.000000
75%,4.000000
max,11.000000


## 시간 누수·기존 final label 일치 감사

입력 마지막 timestamp가 판단봉 `t`이고 진입 및 첫 미래 label timestamp는 정확히 `t+1`인지
검사한다. 기존 v1 artifact가 있으면 동일 표본의 최종 5분 label도 비교한다.

In [6]:
assert metadata['target_net3_by_1m'].sum() == 0  # entry minute min_bid에는 진입 직후 bid가 포함된다.
assert (
    metadata['decision_bar_timestamp'] - metadata['input_start_timestamp']
).dt.total_seconds().eq((SEQUENCE_LENGTH - 1) * 60).all()
for horizon in range(1, HORIZON_MINUTES + 1):
    expected = metadata['entry_timestamp'] + pd.to_timedelta(horizon - 1, unit='m')
    assert metadata[f'future_timestamp_{horizon}'].eq(expected).all()

old_path = RESULT_ROOT / 'scalp_30m_ohlcv_net3_minbid_5m_v1_metadata.parquet'
comparison_summary = {'old_artifact_found': old_path.exists()}
if old_path.exists():
    old = pd.read_parquet(old_path, columns=[
        'session', 'symbol', 'decision_bar_timestamp', 'target_net3_within_5m'
    ])
    joined = metadata.merge(
        old, on=['session', 'symbol', 'decision_bar_timestamp'], how='outer',
        suffixes=('_v2', '_v1'), indicator=True,
    )
    comparison_summary.update({
        'matched_rows': int(joined['_merge'].eq('both').sum()),
        'v2_only_rows': int(joined['_merge'].eq('left_only').sum()),
        'v1_only_rows': int(joined['_merge'].eq('right_only').sum()),
        'label_mismatches': int((
            joined.loc[joined['_merge'].eq('both'), 'target_net3_within_5m_v2'] !=
            joined.loc[joined['_merge'].eq('both'), 'target_net3_within_5m_v1']
        ).sum()),
    })
    assert comparison_summary['v2_only_rows'] == 0
    assert comparison_summary['v1_only_rows'] == 0
    assert comparison_summary['label_mismatches'] == 0

print('time alignment: OK')
display(pd.Series(comparison_summary).to_frame('value'))

time alignment: OK


,value
old_artifact_found,True
matched_rows,72181
v2_only_rows,0
v1_only_rows,0
label_mismatches,0
